# Multimodal RAG with Groq and Free Local Embeddings

This version preserves the original multimodal workflow: PDF structure, tables, figures, searchable summaries, vector retrieval, and image-aware answers.

Embeddings run locally with `BAAI/bge-small-en-v1.5`. Groq handles text and image understanding.

## System dependencies

On macOS, install the PDF/OCR tools once in Terminal:

```bash
brew install poppler tesseract libmagic
uv add unstructured langchain-chroma langchain-community langchain-huggingface sentence-transformers python-dotenv
```

In [ ]:
import base64
import importlib.util
import json
import os
import sys
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

site_packages = Path(sys.executable).parent.parent / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
charset_package = site_packages / 'charset_normalizer'
charset_spec = importlib.util.spec_from_file_location(
    'charset_normalizer',
    charset_package / '__init__.py',
    submodule_search_locations=[str(charset_package)],
)
charset_module = importlib.util.module_from_spec(charset_spec)
sys.modules['charset_normalizer'] = charset_module
charset_spec.loader.exec_module(charset_module)
assert hasattr(charset_module, 'detect')

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from unstructured.chunking.title import chunk_by_title
from unstructured.partition.pdf import partition_pdf

load_dotenv()

current_directory = Path.cwd()
PROJECT_DIRECTORY = (
    current_directory / 'rag-for-beginners'
    if (current_directory / 'rag-for-beginners' / 'docs').is_dir()
    else current_directory
)
PDF_PATH = PROJECT_DIRECTORY / 'docs' / '66---16-06-2021-02-48-53.pdf'
PERSIST_DIRECTORY = PROJECT_DIRECTORY / 'db' / 'chroma_multimodal_groq_bge'
EXPORT_DIRECTORY = PROJECT_DIRECTORY / 'multimodal_exports'
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
GROQ_TEXT_MODEL = 'openai/gpt-oss-120b'
GROQ_VISION_MODEL = os.getenv('GROQ_VISION_MODEL', GROQ_TEXT_MODEL)

ImportError: cannot import name 'get_option' from 'pandas._config' (unknown location)

## 1. Partition the PDF into text, tables, figures, and captions

In [ ]:
def partition_document(file_path):
    if not file_path.is_file():
        raise FileNotFoundError(f'PDF file not found: {file_path}')

    print(f'Partitioning: {file_path.name}')
    elements = partition_pdf(
        filename=str(file_path),
        strategy='hi_res',
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True,
    )
    print(f'Extracted {len(elements)} elements')
    return elements

elements = partition_document(PDF_PATH)
element_types = sorted({type(element).__name__ for element in elements})
images = [element for element in elements if getattr(element, 'category', '') == 'Image']
tables = [element for element in elements if getattr(element, 'category', '') == 'Table']
print(f'Element types: {element_types}')
print(f'Images: {len(images)} | Tables: {len(tables)}')

## 2. Create structure-aware chunks

In [ ]:
chunks = chunk_by_title(
    elements,
    max_characters=3000,
    new_after_n_chars=2400,
    combine_text_under_n_chars=500,
)
print(f'Created {len(chunks)} structure-aware chunks')
print(chunks[0].text[:500])

## 3. Preserve text, table HTML, and image payloads

In [ ]:
def separate_content_types(chunk):
    content = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text'],
    }
    original_elements = getattr(getattr(chunk, 'metadata', None), 'orig_elements', [])
    for element in original_elements:
        element_type = type(element).__name__
        if element_type == 'Table':
            content['types'].append('table')
            table_html = getattr(
                getattr(element, 'metadata', None),
                'text_as_html',
                getattr(element, 'text', ''),
            )
            content['tables'].append(table_html)
        elif element_type == 'Image':
            image_base64 = getattr(getattr(element, 'metadata', None), 'image_base64', None)
            if image_base64:
                content['types'].append('image')
                content['images'].append(image_base64)
    content['types'] = sorted(set(content['types']))
    return content

content_preview = [separate_content_types(chunk) for chunk in chunks[:3]]
for index, item in enumerate(content_preview, 1):
    print(f'Chunk {index}: {item["types"]}, tables={len(item["tables"])}, images={len(item["images"])}')

## 4. Use Groq to create searchable multimodal summaries

In [ ]:
def call_groq(prompt_text, images=None, model=GROQ_TEXT_MODEL):
    groq_api_key = os.getenv('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError('GROQ_API_KEY is not set in the environment or .env file')

    images = images or []
    if images:
        prompt_text += (
            f'\n\nNote: this Groq account has no vision model enabled. '
            f'{len(images)} image payload(s) were detected and preserved in metadata, '
            'but their pixels cannot be visually analyzed in this run.'
        )

    request = Request(
        'https://api.groq.com/openai/v1/chat/completions',
        data=json.dumps({
            'model': model,
            'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': prompt_text}]}],
            'temperature': 0,
            'max_tokens': 1200,
        }).encode('utf-8'),
        headers={
            'Authorization': f'Bearer {groq_api_key}',
            'Content-Type': 'application/json',
            'User-Agent': 'genai-multimodal-rag/1.0',
        },
        method='POST',
    )
    try:
        with urlopen(request) as response:
            result = json.load(response)
    except HTTPError as error:
        details = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Groq request failed with HTTP {error.code}: {details}') from error
    return result['choices'][0]['message']['content']

def create_searchable_summary(content):
    if not content['tables'] and not content['images']:
        return content['text']

    prompt = f'''Create a detailed searchable description of this document chunk.
Cover key facts, numbers, concepts, table values, figure captions, and likely questions.

TEXT:
{content['text']}

TABLES:
{json.dumps(content['tables'], ensure_ascii=False)}

Detected image count: {len(content['images'])}

Return only the searchable description.'''
    return call_groq(prompt, content['images'], GROQ_TEXT_MODEL)

In [ ]:
def process_chunks(chunks):
    processed = []
    for index, chunk in enumerate(chunks, 1):
        content = separate_content_types(chunk)
        enhanced_content = create_searchable_summary(content)
        processed.append(Document(
            page_content=enhanced_content,
            metadata={
                'source': str(PDF_PATH),
                'chunk_id': index,
                'original_content': json.dumps({
                    'raw_text': content['text'],
                    'tables_html': content['tables'],
                    'images_base64': content['images'],
                }, ensure_ascii=False),
            },
        ))
        if index % 10 == 0 or index == len(chunks):
            print(f'Processed {index}/{len(chunks)} chunks')
    return processed

processed_chunks = process_chunks(chunks)
print(f'Processed documents: {len(processed_chunks)}')

## 5. Embed locally and store in Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

db = Chroma.from_documents(
    documents=processed_chunks,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
    collection_metadata={'hnsw:space': 'cosine'},
)
print(f'Chroma database ready at {PERSIST_DIRECTORY}')

## 6. Retrieve multimodal chunks and export results

In [ ]:
def export_documents(documents, filename):
    EXPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    data = []
    for index, document in enumerate(documents, 1):
        data.append({
            'chunk_id': index,
            'enhanced_content': document.page_content,
            'metadata': {
                'source': document.metadata.get('source'),
                'original_content': json.loads(document.metadata.get('original_content', '{}')),
            },
        })
    output_path = EXPORT_DIRECTORY / filename
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Exported {len(data)} chunks to {output_path}')
    return data

query = 'What are the main ideas, architecture, and important results discussed in this document?'
retrieved_chunks = db.similarity_search(query, k=5)
export_documents(retrieved_chunks, 'multimodal_rag_results.json')

## 7. Generate a final answer with text, tables, and images

In [ ]:
def generate_final_answer(documents, question):
    prompt_parts = [f'''Answer this question using only the retrieved PDF content: {question}

Use the text, tables, and image captions/counts. If the evidence is insufficient, say you do not know.

RETRIEVED CONTENT:
''']
    images = []
    for index, document in enumerate(documents, 1):
        original = json.loads(document.metadata.get('original_content', '{}'))
        prompt_parts.append(f'--- Chunk {index} ---\n')
        prompt_parts.append(f'TEXT:\n{original.get("raw_text", "")}\n')
        for table_index, table in enumerate(original.get('tables_html', []), 1):
            prompt_parts.append(f'TABLE {table_index}:\n{table}\n')
        images.extend(original.get('images_base64', []))
    prompt_parts.append(
        f'\nDetected {len(images)} retrieved image payload(s). '
        'Image pixels are preserved but are not visually analyzed because this Groq account has no vision model enabled. '
        'Give a clear, accurate answer with the important facts and numbers.'
    )
    return call_groq('\n'.join(prompt_parts), images, GROQ_TEXT_MODEL)

print(generate_final_answer(retrieved_chunks, query))